# N-ATLaS Training Input Boundary & Formatter Smoke Test

This notebook verifies that `NAtlasFormatter` converts canonical Foundry `DatasetRecord` objects into conversational messages (`user` -> `input`, `assistant` -> `target`) and applies the official `NCAIR1/N-ATLaS` chat template (`tokenize=False`, `add_generation_prompt=False`) on actual hardware/environment.

In [ ]:
# Install dependencies if needed in Kaggle environment
!pip install -q -U transformers accelerate pydantic huggingface_hub

In [ ]:
import sys
from pathlib import Path

# If running inside cloned repository or root
repo_root = Path.cwd()
if (repo_root / "foundry").exists() and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import dataclass
from pydantic import BaseModel, Field

try:
    from api.schemas.dataset import DatasetRecord
    from foundry.training.natlas_formatter import NAtlasFormatter, FormattedTrainingExample
    print("Successfully imported DatasetRecord and NAtlasFormatter from repository.")
except ImportError:
    print("Falling back to self-contained classes for standalone Kaggle run:")
    
    class DatasetRecord(BaseModel):
        id: str = Field(..., min_length=1)
        input: str = Field(..., min_length=1)
        target: str = Field(..., min_length=1)
        language_code: str = Field(..., min_length=2, max_length=8)

    @dataclass(frozen=True)
    class FormattedTrainingExample:
        record_id: str
        language_code: str
        messages: list[dict[str, str]]
        formatted_text: str

    class NAtlasFormatter:
        def __init__(self, tokenizer):
            self.tokenizer = tokenizer

        def _validate_record(self, record: DatasetRecord) -> None:
            if not record.input.strip():
                raise ValueError("DatasetRecord.input cannot be empty")
            if not record.target.strip():
                raise ValueError("DatasetRecord.target cannot be empty")
            if not record.language_code.strip():
                raise ValueError("DatasetRecord.language_code cannot be empty")

        def build_messages(self, record: DatasetRecord) -> list[dict[str, str]]:
            return [
                {"role": "user", "content": record.input},
                {"role": "assistant", "content": record.target},
            ]

        def format(self, record: DatasetRecord) -> FormattedTrainingExample:
            self._validate_record(record)
            messages = self.build_messages(record)
            formatted_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            return FormattedTrainingExample(
                record_id=record.id,
                language_code=record.language_code,
                messages=messages,
                formatted_text=formatted_text,
            )
    print("Self-contained NAtlasFormatter ready.")

In [ ]:
# Authenticate hugging face
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

print("Hugging Face authentication completed.")

In [ ]:
# Load N-ATLaS tokenizer from Hugging Face
from transformers import AutoTokenizer

MODEL_NAME = "NCAIR1/N-ATLaS"
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Chat template defined:", tokenizer.chat_template is not None)

In [ ]:
# Create sample Igala DatasetRecord
record = DatasetRecord(
    id="formatter-smoke-001",
    input="How are you?",
    target="Abe ele?",
    language_code="igl",
)

formatter = NAtlasFormatter(tokenizer)
result = formatter.format(record)

print("Record ID:", result.record_id)
print("Language:", result.language_code)
print("Messages:")
print(result.messages)
print()
print("Formatted text:")
print(result.formatted_text)

In [ ]:
# Inspect tokenization count and special tokens
token_output = tokenizer(result.formatted_text)
token_ids = token_output["input_ids"]
print("Token count:", len(token_ids))
print("Token IDs:", token_ids)
print("Decoded representation with tokens:")
for tid in token_ids:
    print(f"  {tid} -> {repr(tokenizer.decode([tid]))}")

In [ ]:
# Multi-language verification test (Igala, Yoruba, Igbo, Hausa)
records = [
    DatasetRecord(id="rec-igl", input="Good morning", target="Ólódù", language_code="igl"),
    DatasetRecord(id="rec-yor", input="Good morning", target="Ẹ ku aarọ", language_code="yor"),
    DatasetRecord(id="rec-ibo", input="Good morning", target="Ụtụtụ ọma", language_code="ibo"),
    DatasetRecord(id="rec-hau", input="Good morning", target="Ina kwana", language_code="hau"),
]

for rec in records:
    res = formatter.format(rec)
    print(f"[{res.language_code}] {res.record_id} -> tokens: {len(tokenizer(res.formatted_text)['input_ids'])}")
    print("   Formatted:", repr(res.formatted_text))
    print("-" * 60)